# GPT-2 作为原始文本语义编码器

## 读取数据文件

In [5]:
import pandas as pd

# 设置Pandas显示选项以确保打印时不省略任何内容
pd.set_option('display.max_rows', None)  # 显示所有行
pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.width', None)  # 宽度自适应，不限制输出宽度
pd.set_option('display.max_colwidth', None)  # 打印字段内容时，不限制最大列宽

# 读取CSV文件
df = pd.read_csv('feature_user_dna_GLTR_FDGPT_label_results.csv')

# 打印行数
print("行数:", len(df))

# 打印列名
print("\n列名:")
print(df.columns.tolist())

# 打印各列的数据类型
print("\n各列的数据类型:")
print(df.dtypes)

# 打印整个DataFrame的内容
print("\n数据内容:")
print(df.head())

行数: 2280

列名:
['user_id', 'social_dna', 'original_dna_size', 'compressed_dna_size', 'compression_ratio', 'follow_request_sent', 'has_extended_profile', 'profile_use_background_image', 'default_profile_image', 'id', 'profile_background_image_url_https', 'verified', 'translator_type', 'profile_text_color', 'profile_image_url_https', 'profile_sidebar_fill_color', 'followers_count', 'profile_sidebar_border_color', 'id_str', 'profile_background_color', 'listed_count', 'is_translation_enabled', 'utc_offset', 'statuses_count', 'description', 'friends_count', 'location', 'profile_link_color', 'profile_image_url', 'following', 'geo_enabled', 'profile_banner_url', 'profile_background_image_url', 'screen_name', 'lang', 'profile_background_tile', 'favourites_count', 'name', 'notifications', 'url', 'created_at', 'contributors_enabled', 'time_zone', 'protected', 'default_profile', 'is_translator', 'entities.url.urls', 'entities.description.urls', 'withheld_in_countries', 'total_tweets_x', 'GLTR_avg_

In [3]:
# 打印各列的数据类型
print("\n各列的数据类型:")
print(df.dtypes)


各列的数据类型:
user_id                           int64
social_dna                       object
original_dna_size                 int64
compressed_dna_size               int64
compression_ratio               float64
                                 ...   
FDGPT_min_tokens                float64
FDGPT_high_prob_ratio           float64
FDGPT_strict_criterion_ratio    float64
label                            object
label_10                          int64
Length: 77, dtype: object


## 模型

In [2]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer, GPT2Model, AdamW, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, balanced_accuracy_score, confusion_matrix, classification_report
)
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# ----------------------------
# 1. 加载数据
# ----------------------------
df = pd.read_csv('feature_user_dna_GLTR_FDGPT_label_results.csv')
y = df['label_10'].values

# ----------------------------
# 2. 构建文本输入
# ----------------------------
text_columns = ['description', 'location', 'name']
df['combined_text'] = df[text_columns].fillna('').astype(str).apply(
    lambda row: ' '.join([s.strip() for s in row if s.strip() != '']), axis=1
)
texts = df['combined_text'].tolist()

# ----------------------------
# 3. 构建行为特征
# ----------------------------
base_social = [
    'followers_count', 'friends_count', 'statuses_count', 'favourites_count',
    'listed_count', 'verified', 'default_profile', 'protected',
    'geo_enabled', 'profile_use_background_image', 'default_profile_image',
    'has_extended_profile', 'follow_request_sent', 'is_translation_enabled',
    'contributors_enabled', 'is_translator', 'profile_background_tile'
]

dna_features = ['original_dna_size', 'compressed_dna_size', 'compression_ratio']

gltr_features = [
    'GLTR_avg_bert_prob', 'GLTR_std_bert_prob', 'GLTR_max_bert_prob', 'GLTR_min_bert_prob',
    'GLTR_avg_gpt2_prob', 'GLTR_std_gpt2_prob', 'GLTR_max_gpt2_prob', 'GLTR_min_gpt2_prob',
    'GLTR_bert_high_prob_ratio', 'GLTR_gpt2_high_prob_ratio'
]

fdgpt_features = [
    'FDGPT_avg_prob', 'FDGPT_std_prob', 'FDGPT_max_prob', 'FDGPT_min_prob',
    'FDGPT_avg_criterion', 'FDGPT_std_criterion', 'FDGPT_max_criterion', 'FDGPT_min_criterion',
    'FDGPT_avg_tokens', 'FDGPT_std_tokens', 'FDGPT_max_tokens', 'FDGPT_min_tokens',
    'FDGPT_high_prob_ratio', 'FDGPT_strict_criterion_ratio'
]

behavior_cols = base_social + dna_features + gltr_features + fdgpt_features
behavior_cols = [col for col in behavior_cols if col in df.columns]

print(f"✅ 使用的行为特征数量: {len(behavior_cols)}")

X_behavior = df[behavior_cols].copy()
bool_cols = X_behavior.select_dtypes(include='bool').columns
X_behavior[bool_cols] = X_behavior[bool_cols].astype(int)
X_behavior = X_behavior.fillna(0).values
scaler = StandardScaler()
X_behavior = scaler.fit_transform(X_behavior)

# ----------------------------
# 4. 划分数据集
# ----------------------------
X_temp_texts, X_text_test, X_temp_behav, X_behav_test, y_temp, y_test = train_test_split(
    texts, X_behavior, y, test_size=0.2, random_state=42, stratify=y
)
X_text_train, X_text_val, X_behav_train, X_behav_val, y_train, y_val = train_test_split(
    X_temp_texts, X_temp_behav, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f"训练集大小: {len(y_train)} | 验证集: {len(y_val)} | 测试集: {len(y_test)}")

# ----------------------------
# 5. Dataset
# ----------------------------
class SocialBotDataset(Dataset):
    def __init__(self, texts, behaviors, labels, tokenizer, max_len=256):
        self.texts = texts
        self.behaviors = behaviors
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'behavior': torch.tensor(self.behaviors[idx], dtype=torch.float),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ----------------------------
# 6. 模型定义（使用 last token 表示）
# ----------------------------
class GPT2SocialBotDetector(nn.Module):
    def __init__(self, gpt2_model_name='gpt2', behavior_dim=47, gpt2_emb_dim=768, hidden_dim=128):
        super().__init__()
        self.gpt2 = GPT2Model.from_pretrained(gpt2_model_name)
        # 不再冻结 GPT-2

        self.mlp_behavior = nn.Sequential(
            nn.Linear(behavior_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.fuse = nn.Sequential(
            nn.Linear(gpt2_emb_dim + hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_dim, 2)
        )

    def forward(self, input_ids, attention_mask, behavior):
        outputs = self.gpt2(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state  # [B, L, H]

        # 使用最后一个有效 token 的表示（GPT-2 无 [CLS]）
        seq_lengths = attention_mask.sum(dim=1) - 1  # [B]
        batch_indices = torch.arange(last_hidden.size(0), device=last_hidden.device)
        gpt2_emb = last_hidden[batch_indices, seq_lengths]  # [B, H]

        behav_emb = self.mlp_behavior(behavior)
        fused = torch.cat([gpt2_emb, behav_emb], dim=1)
        logits = self.fuse(fused)
        return logits

# ----------------------------
# 7. 训练设置
# ----------------------------
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MAX_LEN = 256  # 可根据显存调整为 128
BATCH_SIZE = 8  # 若显存充足可设为 16
EPOCHS = 10
PATIENCE = 3

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

train_dataset = SocialBotDataset(X_text_train, X_behav_train, y_train, tokenizer, MAX_LEN)
val_dataset = SocialBotDataset(X_text_val, X_behav_val, y_val, tokenizer, MAX_LEN)
test_dataset = SocialBotDataset(X_text_test, X_behav_test, y_test, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

model = GPT2SocialBotDetector(
    behavior_dim=len(behavior_cols),
).to(DEVICE)

# 分层学习率
gpt2_params = list(model.gpt2.parameters())
mlp_params = list(model.mlp_behavior.parameters()) + list(model.fuse.parameters())
optimizer = AdamW([
    {'params': gpt2_params, 'lr': 1e-5},
    {'params': mlp_params, 'lr': 2e-4}
])

criterion = nn.CrossEntropyLoss()
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

# ----------------------------
# 8. 评估函数
# ----------------------------
def evaluate_metrics(data_loader):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            behavior = batch['behavior'].to(DEVICE)
            labels = batch['label'].to(DEVICE)

            logits = model(input_ids, attention_mask, behavior)
            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = logits.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    prc = precision_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    roc = roc_auc_score(all_labels, all_probs)
    bep = balanced_accuracy_score(all_labels, all_preds)
    cm = confusion_matrix(all_labels, all_preds)
    return acc, prc, rec, f1, roc, bep, cm

# ----------------------------
# 9. 训练循环（带早停）
# ----------------------------
best_val_loss = float('inf')
trigger_times = 0

for epoch in range(EPOCHS):
    # 训练
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]"):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        behavior = batch['behavior'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask, behavior)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader)

    # 验证
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            behavior = batch['behavior'].to(DEVICE)
            labels = batch['label'].to(DEVICE)

            logits = model(input_ids, attention_mask, behavior)
            loss = criterion(logits, labels)
            val_loss += loss.item()
    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # 早停 & 保存最佳模型
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        trigger_times = 0
        torch.save(model.state_dict(), 'best_gpt2_socialbot_model.pt')
    else:
        trigger_times += 1
        if trigger_times >= PATIENCE:
            print("🛑 Early stopping triggered.")
            break

# ----------------------------
# 10. 最终测试评估
# ----------------------------
print("\n" + "="*60)
print("🔍 加载最佳模型进行最终测试评估...")
model.load_state_dict(torch.load('best_gpt2_socialbot_model.pt'))

acc, prc, rec, f1, roc, bep, cm = evaluate_metrics(test_loader)

print(f"✅ Accuracy : {acc:.4f}")
print(f"✅ Precision: {prc:.4f}")
print(f"✅ Recall   : {rec:.4f}")
print(f"✅ F1-score : {f1:.4f}")
print(f"✅ ROC-AUC  : {roc:.4f}")
print(f"✅ Balanced Accuracy (BEP): {bep:.4f}")
print("✅ Confusion Matrix:")
print(cm)

# 可选：详细分类报告
print("\n📊 详细分类报告:")
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        behavior = batch['behavior'].to(DEVICE)
        labels = batch['label'].to(DEVICE)
        logits = model(input_ids, attention_mask, behavior)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
print(classification_report(all_labels, all_preds, target_names=['human', 'bot']))

2025-09-25 16:55:41.511426: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-25 16:55:41.513866: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-09-25 16:55:41.570623: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-25 16:55:42.432286: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


✅ 使用的行为特征数量: 44
训练集大小: 1368 | 验证集: 456 | 测试集: 456


Epoch 1 [Val]: 100%|██████████| 57/57 [00:06<00:00,  9.07it/s]


Epoch 1/10 | Train Loss: 0.9563 | Val Loss: 0.2949


Epoch 2 [Val]: 100%|██████████| 57/57 [00:06<00:00,  8.94it/s]


...

Epoch 5/10 | Train Loss: 0.0482 | Val Loss: 0.1629


Epoch 6 [Val]: 100%|██████████| 57/57 [00:06<00:00,  8.81it/s]


Epoch 6/10 | Train Loss: 0.0182 | Val Loss: 0.1970


Epoch 7 [Val]: 100%|██████████| 57/57 [00:06<00:00,  8.83it/s]


Epoch 7/10 | Train Loss: 0.0132 | Val Loss: 0.1983
🛑 Early stopping triggered.

🔍 加载最佳模型进行最终测试评估...
✅ Accuracy : 0.9846
✅ Precision: 0.9825
✅ Recall   : 0.9868
✅ F1-score : 0.9847
✅ ROC-AUC  : 0.9986
✅ Balanced Accuracy (BEP): 0.9846
✅ Confusion Matrix:
[[224   4]
 [  3 225]]

📊 详细分类报告:
              precision    recall  f1-score   support

       human       0.99      0.98      0.98       228
         bot       0.98      0.99      0.98       228

    accuracy                           0.98       456
   macro avg       0.98      0.98      0.98       456
weighted avg       0.98      0.98      0.98       456



## 跨数据集测试

### BotSim

#### 查看数据

In [4]:
import pandas as pd

# 设置Pandas显示选项以确保打印时不省略任何内容
pd.set_option('display.max_rows', None)  # 显示所有行
pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.width', None)  # 宽度自适应，不限制输出宽度
pd.set_option('display.max_colwidth', None)  # 打印字段内容时，不限制最大列宽

# 读取CSV文件
df = pd.read_csv('feature_user_dna_label_GLTR_FGPT_results.csv')

# 打印行数
print("行数:", len(df))

# 打印列名
print("\n列名:")
print(df.columns.tolist())

# 打印各列的数据类型
print("\n各列的数据类型:")
print(df.dtypes)

# 打印整个DataFrame的内容
print("\n数据内容:")
print(df.head())

行数: 2906

列名:
['user_id', 'name', 'description', 'submission_num', 'comment_num', 'character_setting', 'comment_num_1', 'comment_num_2', 'subreddit', 'class', 'social_dna', 'original_dna_size', 'compressed_dna_size', 'compression_ratio', 'total_tweets_x', 'GLTR_avg_bert_prob', 'GLTR_std_bert_prob', 'GLTR_max_bert_prob', 'GLTR_min_bert_prob', 'GLTR_avg_gpt2_prob', 'GLTR_std_gpt2_prob', 'GLTR_max_gpt2_prob', 'GLTR_min_gpt2_prob', 'GLTR_bert_high_prob_ratio', 'GLTR_gpt2_high_prob_ratio', 'total_tweets_y', 'FDGPT_avg_prob', 'FDGPT_std_prob', 'FDGPT_max_prob', 'FDGPT_min_prob', 'FDGPT_avg_criterion', 'FDGPT_std_criterion', 'FDGPT_max_criterion', 'FDGPT_min_criterion', 'FDGPT_avg_tokens', 'FDGPT_std_tokens', 'FDGPT_max_tokens', 'FDGPT_min_tokens', 'FDGPT_high_prob_ratio', 'FDGPT_strict_criterion_ratio']

各列的数据类型:
user_id                          object
name                             object
description                      object
submission_num                  float64
comment_num          

In [6]:
import pandas as pd

# 读取 CSV 文件
df = pd.read_csv('feature_user_dna_label_GLTR_FGPT_results.csv')

# 统计 class 列中 0 和 1 的数量
counts = df['class'].value_counts()

# 输出结果
print("class 列统计：")
print(f"0 的数量: {counts.get(0, 0)}")
print(f"1 的数量: {counts.get(1, 0)}")

class 列统计：
0 的数量: 1906
1 的数量: 1000


#### 模型训练

In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer, GPT2Model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, balanced_accuracy_score, confusion_matrix, classification_report
)
from sklearn.utils.class_weight import compute_class_weight

# ----------------------------
# 1. 加载数据 + 下采样多数类（class=0）
# ----------------------------
df = pd.read_csv('feature_user_dna_label_GLTR_FGPT_results.csv')

df_0 = df[df['class'] == 0]
df_1 = df[df['class'] == 1]

print(f"原始 class=0 数量: {len(df_0)}")
print(f"原始 class=1 数量: {len(df_1)}")

df_0_sampled = df_0.sample(n=1000, random_state=42)
df_balanced = pd.concat([df_0_sampled, df_1], ignore_index=True)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"✅ 平衡后总样本数: {len(df_balanced)}")
print("✅ 平衡后 class 分布:")
print(df_balanced['class'].value_counts().sort_index())

# ----------------------------
# 2. 构建文本输入：拼接 name + description
# ----------------------------
text_columns = ['name', 'description']
df_balanced['combined_text'] = df_balanced[text_columns].fillna('').astype(str).apply(
    lambda row: ' '.join([s.strip() for s in row if s.strip() != '']), axis=1
)
texts = df_balanced['combined_text'].tolist()

# ----------------------------
# 3. 构建行为特征列（仅原始数值特征，不含文本统计）
# ----------------------------
base_social = [
    'submission_num', 'comment_num', 'comment_num_1', 'comment_num_2',
    'total_tweets_x'
]

dna_features = [
    'original_dna_size', 'compressed_dna_size', 'compression_ratio'
]

gltr_features = [
    'GLTR_avg_bert_prob', 'GLTR_std_bert_prob', 'GLTR_max_bert_prob', 'GLTR_min_bert_prob',
    'GLTR_avg_gpt2_prob', 'GLTR_std_gpt2_prob', 'GLTR_max_gpt2_prob', 'GLTR_min_gpt2_prob',
    'GLTR_bert_high_prob_ratio', 'GLTR_gpt2_high_prob_ratio'
]

fdgpt_features = [
    'FDGPT_avg_prob', 'FDGPT_std_prob', 'FDGPT_max_prob', 'FDGPT_min_prob',
    'FDGPT_avg_criterion', 'FDGPT_std_criterion', 'FDGPT_max_criterion', 'FDGPT_min_criterion',
    'FDGPT_avg_tokens', 'FDGPT_std_tokens', 'FDGPT_max_tokens', 'FDGPT_min_tokens',
    'FDGPT_high_prob_ratio', 'FDGPT_strict_criterion_ratio'
]

behavior_cols = base_social + dna_features + gltr_features + fdgpt_features
behavior_cols = [col for col in behavior_cols if col in df_balanced.columns]

print(f"✅ 使用的行为特征数量: {len(behavior_cols)}")
print("行为特征示例:", behavior_cols[:5])

# 提取行为特征并处理缺失值
X_behavior = df_balanced[behavior_cols].copy()
X_behavior = X_behavior.fillna(0).values
scaler = StandardScaler()
X_behavior = scaler.fit_transform(X_behavior)

# 标签
y = df_balanced['class'].values

# ----------------------------
# 4. 划分数据集（训练:验证:测试 = 6:2:2，分层抽样）
# ----------------------------
(
    X_temp_texts, X_text_test,
    X_temp_behav, X_behav_test,
    y_temp, y_test
) = train_test_split(
    texts, X_behavior, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

(
    X_text_train, X_text_val,
    X_behav_train, X_behav_val,
    y_train, y_val
) = train_test_split(
    X_temp_texts, X_temp_behav, y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp
)

print(f"✅ 训练集大小: {len(y_train)} (class 0: {sum(y_train == 0)}, class 1: {sum(y_train == 1)})")
print(f"✅ 验证集大小: {len(y_val)} (class 0: {sum(y_val == 0)}, class 1: {sum(y_val == 1)})")
print(f"✅ 测试集大小: {len(y_test)} (class 0: {sum(y_test == 0)}, class 1: {sum(y_test == 1)})")

# ----------------------------
# 5. Dataset 定义
# ----------------------------
class SocialBotDataset(Dataset):
    def __init__(self, texts, behaviors, labels, tokenizer, max_len=128):
        self.texts = texts
        self.behaviors = behaviors
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'behavior': torch.tensor(self.behaviors[idx], dtype=torch.float),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ----------------------------
# 6. 模型定义（解冻 GPT-2 后6层）
# ----------------------------
class GPT2SocialBotDetector(nn.Module):
    def __init__(self, gpt2_model_name='gpt2', behavior_dim=None, gpt2_emb_dim=768, hidden_dim=128):
        super().__init__()
        self.gpt2 = GPT2Model.from_pretrained(gpt2_model_name)
        
        # 冻结前6层，微调后6层（GPT-2 共12层）
        for i, layer in enumerate(self.gpt2.h):
            for param in layer.parameters():
                param.requires_grad = (i >= 6)  # i=0~11，后6层可训练
        
        # 冻结 token 和 position embeddings
        for param in self.gpt2.wte.parameters():
            param.requires_grad = False
        for param in self.gpt2.wpe.parameters():
            param.requires_grad = False

        self.mlp_behavior = nn.Sequential(
            nn.Linear(behavior_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.fuse = nn.Sequential(
            nn.Linear(gpt2_emb_dim + hidden_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, input_ids, attention_mask, behavior):
        outputs = self.gpt2(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state

        # Mean pooling with attention mask
        attention_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden.size())
        sum_embeddings = torch.sum(last_hidden * attention_mask_expanded, dim=1)
        sum_mask = torch.clamp(attention_mask.sum(dim=1).unsqueeze(1), min=1e-9)
        gpt2_emb = sum_embeddings / sum_mask

        behav_emb = self.mlp_behavior(behavior)
        fused = torch.cat([gpt2_emb, behav_emb], dim=1)
        logits = self.fuse(fused)
        return logits

# ----------------------------
# 7. 训练设置
# ----------------------------
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 15

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

train_dataset = SocialBotDataset(X_text_train, X_behav_train, y_train, tokenizer, MAX_LEN)
val_dataset = SocialBotDataset(X_text_val, X_behav_val, y_val, tokenizer, MAX_LEN)
test_dataset = SocialBotDataset(X_text_test, X_behav_test, y_test, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

model = GPT2SocialBotDetector(
    behavior_dim=len(behavior_cols),
).to(DEVICE)

# ✅ 使用类别加权损失（基于训练集）
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)  # 更适合微调的学习率

# ----------------------------
# 8. 评估函数
# ----------------------------
def evaluate_metrics(data_loader, dataset_name="Test"):
    model.eval()
    all_preds = []
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for batch in tqdm(data_loader, desc=f"Evaluating {dataset_name}"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            behavior = batch['behavior'].to(DEVICE)
            labels = batch['label'].to(DEVICE)

            logits = model(input_ids, attention_mask, behavior)
            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = logits.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    prc = precision_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    roc = roc_auc_score(all_labels, all_probs)
    bep = balanced_accuracy_score(all_labels, all_preds)
    cm = confusion_matrix(all_labels, all_preds)

    return acc, prc, rec, f1, roc, bep, cm, all_labels, all_preds

# ----------------------------
# 9. 训练循环（带早停）
# ----------------------------
best_val_loss = float('inf')
patience = 3
trigger_times = 0

for epoch in range(EPOCHS):
    # 训练阶段
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]"):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        behavior = batch['behavior'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask, behavior)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader)

    # 验证阶段
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            behavior = batch['behavior'].to(DEVICE)
            labels = batch['label'].to(DEVICE)

            logits = model(input_ids, attention_mask, behavior)
            loss = criterion(logits, labels)
            val_loss += loss.item()
    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # 早停 + 保存最佳模型
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        trigger_times = 0
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        trigger_times += 1
        if trigger_times >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

# ----------------------------
# 10. 最终评估（测试集）
# ----------------------------
print("\n" + "="*50)
print("📊 最终测试集性能指标:")

# 加载验证集上表现最好的模型
model.load_state_dict(torch.load('best_model.pth'))
acc, prc, rec, f1, roc, bep, cm, all_labels, all_preds = evaluate_metrics(test_loader, "Test")

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prc:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {roc:.4f}")
print(f"BEP      : {bep:.4f}")
print("Confusion Matrix:")
print(cm)

print("\n详细分类报告:")
print(classification_report(all_labels, all_preds, target_names=['human', 'bot']))

2025-10-20 20:52:16.383153: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-20 20:52:16.470397: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-20 20:52:17.102804: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-20 20:52:18.408500: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


原始 class=0 数量: 1906
原始 class=1 数量: 1000
✅ 平衡后总样本数: 2000
✅ 平衡后 class 分布:
class
0    1000
1    1000
Name: count, dtype: int64
✅ 使用的行为特征数量: 32
行为特征示例: ['submission_num', 'comment_num', 'comment_num_1', 'comment_num_2', 'total_tweets_x']
✅ 训练集大小: 1200 (class 0: 600, class 1: 600)
✅ 验证集大小: 400 (class 0: 200, class 1: 200)
✅ 测试集大小: 400 (class 0: 200, class 1: 200)


Epoch 1 [Val]: 100%|██████████| 25/25 [00:02<00:00,  9.21it/s]


Epoch 1/15 | Train Loss: 0.7174 | Val Loss: 0.5886


Epoch 2 [Val]: 100%|██████████| 25/25 [00:02<00:00,  9.21it/s]


...

Epoch 9/15 | Train Loss: 0.0238 | Val Loss: 0.0553


Epoch 10 [Val]: 100%|██████████| 25/25 [00:02<00:00,  8.75it/s]
/tmp/ipykernel_8150/2740563078.py:316: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(to

Epoch 10/15 | Train Loss: 0.0169 | Val Loss: 0.0438
Early stopping at epoch 10

📊 最终测试集性能指标:


Evaluating Test: 100%|██████████| 25/25 [00:02<00:00,  8.68it/s]

Accuracy : 0.9750
Precision: 0.9567
Recall   : 0.9950
F1-score : 0.9755
ROC-AUC  : 0.9977
BEP      : 0.9750
Confusion Matrix:
[[191   9]
 [  1 199]]

详细分类报告:
              precision    recall  f1-score   support

       human       0.99      0.95      0.97       200
         bot       0.96      0.99      0.98       200

    accuracy                           0.97       400
   macro avg       0.98      0.97      0.97       400
weighted avg       0.98      0.97      0.97       400

